# 第四階段：計算相鄰圓心的距離

**輸入**：第三階段修準後的圓（圓心 x, y 和半徑 r）
**輸出**：
- 每一對相鄰圓的圓心距離（`pairs.csv`）
- 每個圓的統計：有幾個鄰居、最近的鄰居多遠、平均多遠（`per_circle.csv`）
- 整張圖的總結（印在畫面上，也存成 `summary.json`）
- 一張把相鄰的圓用線連起來的圖（`neighbors.jpg`）

## 怎麼用
1. 改「設定」那格的 `IMAGE_PATH` 和 `OUTPUT_DIR`
2. `CIRCLES` 留 `None`：自動讀第三階段存的結果（`.tmp/<圖片檔名>/refine/summary.json`）
3. 從上到下依序執行

## 做了什麼
1. **兩點距離**：用畢氏定理算兩個圓心的距離 √((x1−x2)² + (y1−y2)²)
2. **兩兩配對**：每兩個圓都算一次，得到一張 N × N 的距離表
3. **判斷鄰居**（兩種方式擇一）：
   - `"touch"`：距離 < FACTOR × (半徑1 + 半徑2) 就算相鄰。兩個圓剛好碰在一起時，距離 = 半徑相加
   - `"knn"`：每個圓固定找最近的 K 個

**注意**：前面偵測錯的圓，也會產生錯的距離。這一步準不準，取決於前三個階段找的圓準不準。

In [ ]:
import os, json, csv
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 如果上面出現 ModuleNotFoundError，先執行：
#   pip install opencv-python numpy matplotlib

## 設定（只要改這裡）

In [ ]:
IMAGE_PATH = "your_image.jpg"     # ← 改成你的圖片路徑
OUTPUT_DIR = "output/neighbors"   # ← 結果要存在哪個資料夾
CIRCLES = None                    # ← None = 自動讀第三階段的結果
TMP_DIR = ".tmp"

# 相鄰的定義
MODE = "touch"          # "touch"：靠得夠近就算；"knn"：固定找最近的 K 個
FACTOR = 1.2            # MODE = "touch" 時用：距離 < FACTOR × (r1 + r2) 算相鄰
                        #   1.0 = 剛好相切才算；1.2 = 中間留一點縫也算
K = 4                   # MODE = "knn" 時用：方格排列通常是 4，蜂巢排列通常是 6

# 被圖片邊界切到的圓（partial），圓心可能不準
EXCLUDE_PARTIAL = False # True = 不列入計算；False = 列入，但會標記

In [ ]:
name = os.path.splitext(os.path.basename(IMAGE_PATH))[0]
if CIRCLES is None:
    stage3 = os.path.join(TMP_DIR, name, "refine", "summary.json")
    CIRCLES = json.load(open(stage3))["circles"]
    print(f"從 {stage3} 讀到 {len(CIRCLES)} 個圓")

# 每個圓加上編號（跟第三階段畫圖的編號一樣）
circles = [dict(c_, index=i) for i, c_ in enumerate(CIRCLES)]
if EXCLUDE_PARTIAL:
    before = len(circles)
    circles = [c_ for c_ in circles if not c_.get("partial")]
    print(f"排除被邊界切到的圓：{before} → {len(circles)} 個")
print(f"參與計算的圓：{len(circles)} 個")

## 步驟 1＋2：兩兩之間的距離表
`xy[:, None, :] - xy[None, :, :]` 這一行，會一次算出「每個圓心減每個圓心」，
得到一張 N × N 的表。表格第 i 列第 j 欄，就是圓 i 到圓 j 的距離。
對角線（自己到自己）是 0。

In [ ]:
def distance_table(circles):
    """回傳 N × N 的距離表，dist[i, j] = 第 i 個圓到第 j 個圓的圓心距離"""
    xy = np.array([[c_["x"], c_["y"]] for c_ in circles])
    diff = xy[:, None, :] - xy[None, :, :]        # 每兩點的 (x 差, y 差)
    return np.sqrt((diff ** 2).sum(axis=2))       # 畢氏定理

## 步驟 3：判斷誰是鄰居
回傳一個清單，每一對鄰居只出現一次，格式是 (i, j, 距離)，i 和 j 是在 `circles` 裡的位置。

In [ ]:
def find_neighbors(circles, dist, mode="touch", factor=1.2, k=4):
    n = len(circles)
    r = np.array([c_["r"] for c_ in circles])
    pairs = set()

    if mode == "touch":
        # 門檻表：limit[i, j] = factor × (r_i + r_j)
        limit = factor * (r[:, None] + r[None, :])
        for i in range(n):
            for j in range(i + 1, n):          # i < j，每一對只看一次
                if dist[i, j] < limit[i, j]:
                    pairs.add((i, j))

    elif mode == "knn":
        for i in range(n):
            order = np.argsort(dist[i])        # 由近到遠排序
            for j in order[1:k + 1]:           # 第 0 個是自己，跳過
                pairs.add((min(i, j), max(i, j)))
    else:
        raise ValueError('MODE 只能是 "touch" 或 "knn"')

    return sorted((i, j, float(dist[i, j])) for i, j in pairs)

## 執行

In [ ]:
dist = distance_table(circles)
pairs = find_neighbors(circles, dist, MODE, FACTOR, K)

# ---- 每個圓的統計 ----
neighbors_of = {i: [] for i in range(len(circles))}
for i, j, d in pairs:
    neighbors_of[i].append((j, d))
    neighbors_of[j].append((i, d))

per_circle = []
for i, c_ in enumerate(circles):
    nb = neighbors_of[i]
    ds = [d for _, d in nb]
    per_circle.append({
        "index": c_["index"],
        "x": c_["x"], "y": c_["y"], "r": c_["r"],
        "partial": c_.get("partial", False),
        "n_neighbors": len(nb),
        "nearest_distance": round(min(ds), 2) if ds else None,
        "mean_distance": round(float(np.mean(ds)), 2) if ds else None,
        "neighbors": " ".join(str(circles[j]["index"]) for j, _ in sorted(nb, key=lambda t: t[1])),
    })

# ---- 整張圖的總結 ----
all_d = np.array([d for _, _, d in pairs])
summary = {
    "image": IMAGE_PATH,
    "mode": MODE, "factor": FACTOR, "k": K, "exclude_partial": EXCLUDE_PARTIAL,
    "n_circles": len(circles),
    "n_pairs": len(pairs),
}
if len(all_d):
    summary.update({
        "distance_mean": round(float(all_d.mean()), 2),
        "distance_median": round(float(np.median(all_d)), 2),
        "distance_std": round(float(all_d.std()), 2),
        "distance_min": round(float(all_d.min()), 2),
        "distance_max": round(float(all_d.max()), 2),
    })

for k_, v in summary.items():
    print(f"{k_:18s} {v}")

## 存檔

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 每一對鄰居
with open(os.path.join(OUTPUT_DIR, "pairs.csv"), "w", newline="") as f:
    wr = csv.writer(f)
    wr.writerow(["circle_a", "circle_b", "distance", "xa", "ya", "xb", "yb", "any_partial"])
    for i, j, d in pairs:
        a, b = circles[i], circles[j]
        wr.writerow([a["index"], b["index"], round(d, 2), a["x"], a["y"], b["x"], b["y"],
                     bool(a.get("partial") or b.get("partial"))])

# 每個圓的統計
with open(os.path.join(OUTPUT_DIR, "per_circle.csv"), "w", newline="") as f:
    wr = csv.DictWriter(f, fieldnames=list(per_circle[0].keys()))
    wr.writeheader()
    wr.writerows(per_circle)

# 總結
with open(os.path.join(OUTPUT_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=1)

print("存在：", OUTPUT_DIR)
for fn in sorted(os.listdir(OUTPUT_DIR)):
    print("  ", fn)

## 畫圖
- 紅色圓 = 完整的圓；橙色圓 = 被邊界切到的圓
- 黃線 = 相鄰的兩個圓；白色數字 = 圓心距離（像素）
- 綠色數字 = 圓的編號

In [ ]:
img = cv2.imread(IMAGE_PATH)
H, W = img.shape[:2]
t = max(2, int(max(H, W) / 500))
font = max(0.5, max(H, W) / 2000)

for i, j, d in pairs:
    a = (int(round(circles[i]["x"])), int(round(circles[i]["y"])))
    b = (int(round(circles[j]["x"])), int(round(circles[j]["y"])))
    cv2.line(img, a, b, (0, 255, 255), t)
    mid = ((a[0] + b[0]) // 2, (a[1] + b[1]) // 2)
    cv2.putText(img, f"{d:.0f}", mid, cv2.FONT_HERSHEY_SIMPLEX, font, (255, 255, 255), max(1, t // 2 + 1))

for c_ in circles:
    center = (int(round(c_["x"])), int(round(c_["y"])))
    color = (0, 165, 255) if c_.get("partial") else (0, 0, 255)
    cv2.circle(img, center, int(round(c_["r"])), color, t)
    cv2.putText(img, str(c_["index"]), (center[0] - 4 * t, center[1] - 4 * t),
                cv2.FONT_HERSHEY_SIMPLEX, font * 1.2, (0, 255, 0), max(1, t // 2 + 1))

out_img = os.path.join(OUTPUT_DIR, "neighbors.jpg")
cv2.imwrite(out_img, img)
print("圖：", out_img)

plt.figure(figsize=(10, 10 * H / W))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

## 距離分布
如果圓是規則排列，距離應該集中在一個值附近，直方圖會是一根又高又窄的柱子。
如果有明顯偏離的柱子，通常是偵測錯的圓造成的，可以對照 `pairs.csv` 找出是哪一對。

In [ ]:
if len(all_d):
    plt.figure(figsize=(8, 3.5))
    plt.hist(all_d, bins=20, color="#4a7bd0", edgecolor="white")
    plt.axvline(np.median(all_d), color="#d62728", ls="--", label=f"median {np.median(all_d):.1f}")
    plt.xlabel("center-to-center distance (pixels)")
    plt.ylabel("number of pairs")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "distance_hist.png"), dpi=100)
    plt.show()